# serving-benchmarks on a T4: FP16, FP16 + prefix caching, AWQ + prefix caching, speculative decoding

Qwen2.5-3B-Instruct on vLLM 0.31. Four cold server starts, each measured with the same open-loop tokenbench sweep (0.5 → 8 req/s, 60 s per level, sample prompts behind a shared 560-word system prompt, natural stopping, SLO TTFT ≤ 1 s and TPOT ≤ 60 ms). Then `serving-benchmarks report` turns the sweeps into latency, throughput, goodput and $/M-token curves. METHODOLOGY.md in the repo is the full description. ~10 min install, ~10 min per configuration.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# this repo is private until the results are in: its wheel is embedded in the first cell by the portfolio's kaggle_run.sh
!pip install -q /kaggle/working/wheels/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, serving_benchmarks.cli, servellm.launch, tokenbench
assert torch.cuda.is_available(), "no GPU"
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))

## Four servers, one sweep each

In [ ]:
import json, subprocess, time
from servellm.launch import ServerConfig, VllmServer
from serving_benchmarks.configs import CONFIGS, serve_command
from serving_benchmarks.cli import sweep_args

BASE = "http://127.0.0.1:8000"
done, failed = [], {}
for cfg in CONFIGS:
    print("=" * 80); print(cfg.slug, "|", cfg.name); print(" ".join(serve_command(cfg)))
    server = VllmServer(ServerConfig(model=cfg.model, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, extra_args=cfg.extra_args),
                        log_path=f"results/startup_{cfg.slug}.log")
    t0 = time.time()
    try:
        info = server.start(ready_timeout_s=1500)
        startup = {**vars(info), "startup_s": time.time() - t0, "config": cfg.name, "model": cfg.model, "extra_args": list(cfg.extra_args)}
        json.dump(startup, open(f"results/startup_{cfg.slug}.json", "w"), indent=1, default=str)
        print(f"ready in {startup['startup_s']:.0f} s | weights {info.model_load_gib} GiB | KV cache {info.kv_cache_tokens} tokens | {info.attention_backend}")
        out = f"results/sweep_{cfg.slug}.json"
        r = subprocess.run(sweep_args(BASE, cfg.model, out), capture_output=True, text=True)
        open(f"results/sweep_{cfg.slug}.log", "w").write(r.stdout + r.stderr)
        if r.returncode != 0:
            raise RuntimeError(f"tokenbench exited {r.returncode}: {(r.stderr or r.stdout)[-800:]}")
        for lv in json.load(open(out))["levels"]:
            print(f"  {lv['level']:>4} req/s: {lv['requests']:>4} req, {lv['errors']} err, {lv['output_tokens_per_s']:>6.0f} tok/s, "
                  f"TTFT p50 {lv['ttft']['p50']*1000:>6.0f} ms, ITL p50 {lv['itl']['p50']*1000:>5.1f} ms, SLO pass {lv.get('slo_pass_rate')}")
        done.append(cfg.slug)
    except Exception as e:
        failed[cfg.slug] = repr(e)[:1500]; print("FAILED", cfg.slug, failed[cfg.slug][:500])
    finally:
        server.stop(); time.sleep(5)
json.dump({"done": done, "failed": failed}, open("results/run_status.json", "w"), indent=1)
print("done:", done, "failed:", list(failed))

## Report

In [ ]:
!serving-benchmarks report --results results --gpu-usd-per-hour 0.35 --common-rps 2
from IPython.display import Markdown, Image, display
display(Markdown(open("results/report/summary.md").read()))
for name in ("ttft", "itl", "throughput", "goodput", "cost"):
    display(Image(f"results/report/{name}.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip